In [10]:
import arcpy

def calculate_ellipsoid_area_to_mu(gdb_path):
    """
    批量计算FileGDB中所有面要素类的椭球面积，并将其转换为亩。

    Args:
        gdb_path (str): FileGDB的完整路径。
    """
    try:
        arcpy.env.workspace = gdb_path
        
        # 遍历GDB中的所有要素数据集
        datasets = arcpy.ListDatasets(feature_type='Feature')
        if not datasets:
            datasets = ['']  # 如果没有要素数据集，则在根目录下操作

        for dataset in datasets:
            # 遍历要素数据集或GDB根目录下的所有面要素类
            arcpy.env.workspace = gdb_path + "\\" + dataset if dataset else gdb_path
            feature_classes = arcpy.ListFeatureClasses(feature_type='Polygon')

            for fc in feature_classes:
                print(f"正在处理要素类： {fc}")

                # 检查字段是否存在，如果不存在则添加
                field_name = "椭球面积"
                field_exists = False
                for field in arcpy.ListFields(fc):
                    if field.name == field_name:
                        field_exists = True
                        break
                
                if not field_exists:
                    print(f"  添加字段：{field_name}")
                    arcpy.AddField_management(fc, field_name, "DOUBLE")
                else:
                    print(f"  字段 {field_name} 已存在，跳过添加。")
                    
                # 检查要素类是否为空
                record_count = int(arcpy.management.GetCount(fc)[0])
                if record_count == 0:
                    print(f"  要素类 {fc} 没有记录，跳过面积计算。")
                    print(f"  要素类 {fc} 处理完成。\n")
                    continue # 跳过当前循环的其余部分，继续下一个要素类
                # 计算椭球面积并转换为亩
                print(f"  计算 {fc} 的椭球面积...")
                
                # acre_to_mu = 6.070284643323547  # 1英亩 ≈ 6.07亩
                # 转换系数：平方米到亩（1亩 = 666.666...平方米）
                sq_meter_to_mu = 1 / 666.6666666666666
                
                arcpy.CalculateGeometryAttributes_management(
                    in_features=fc,
                    geometry_property=[[field_name, "AREA_GEODESIC"]],
                    area_unit="SQUARE_METERS",
                    length_unit="",
                    coordinate_system=""
                )

                # 将平方米值转换为亩
                print(f"  将 {fc} 的面积值转换为亩...")
                arcpy.management.CalculateField(
                    in_table=fc,
                    field=field_name,
                    expression=f"!{field_name}! * {sq_meter_to_mu}",
                    expression_type="PYTHON3"
                )
                
                print(f"  要素类 {fc} 处理完成。\n")

    except arcpy.ExecuteError:
        print(f"ArcPy执行错误: {arcpy.GetMessages(2)}")
    except Exception as e:
        print(f"发生了一个错误: {e}")

if __name__ == '__main__':
    # 请修改为你的FileGDB完整路径
    gdb_path = r"E:\成果库\CDData0820_1.gdb" 
    calculate_ellipsoid_area_to_mu(gdb_path)

正在处理要素类： XXA
  字段 椭球面积 已存在，跳过添加。
  计算 XXA 的椭球面积...
  将 XXA 的面积值转换为亩...
  要素类 XXA 处理完成。

正在处理要素类： YLJGA
  字段 椭球面积 已存在，跳过添加。
  计算 YLJGA 的椭球面积...
  将 YLJGA 的面积值转换为亩...
  要素类 YLJGA 处理完成。

正在处理要素类： XZCSGXGZYDA
  字段 椭球面积 已存在，跳过添加。
  计算 XZCSGXGZYDA 的椭球面积...
  将 XZCSGXGZYDA 的面积值转换为亩...
  要素类 XZCSGXGZYDA 处理完成。

正在处理要素类： SWHXCBCTQTBA
  字段 椭球面积 已存在，跳过添加。
  要素类 SWHXCBCTQTBA 没有记录，跳过面积计算。
  要素类 SWHXCBCTQTBA 处理完成。

